In [ ]:
import serial
import socket
import csv
import time
from datetime import datetime

#########################################
# CONFIGURACIÓN (MODIFICAR ESTO)
#########################################
COM_PORT      = "COM3"       # Cambiar según tu PC
BAUD_RATE     = 115200       # Ajustar según tu dispositivo
UDP_IP        = "127.0.0.1"  # g.Recorder en misma PC
UDP_PORT      = 1000         # MISMO QUE CONFIGURES EN g.Recorder
FILE_LOG      = "fuerza_log.csv"
FILE_TRIGGERS = "triggers_log.csv"
#########################################

# Inicializamos puerto serial
ser = serial.Serial(COM_PORT, BAUD_RATE)

# Inicializamos socket UDP
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# Abrimos archivos CSV
with open(FILE_LOG, "w", newline="") as f_data, open(FILE_TRIGGERS, "w", newline="") as f_trig:
    writer_data = csv.writer(f_data)
    writer_trig = csv.writer(f_trig)

    # Cabeceras
    writer_data.writerow(["timestamp", "force"])
    writer_trig.writerow(["timestamp", "tipo_trigger", "udp_value"])

    print(">>> Leyendo puerto serial... (Ctrl+C para terminar)")

    try:
        while True:
            line = ser.readline().decode(errors='ignore').strip()

            # --- Guardamos TODOS los datos brutos ---
            now = time.time()    # timestamp unix
            writer_data.writerow([now, line])

            # --- Mostramos en consola ---
            print(f"[{datetime.now().strftime('%H:%M:%S')}] {line}")

            # --- Detectar y enviar trigger ---
            if "marca" in line.lower():
                print(">>> TRIGGER DETECTADO – Enviando a g.Recorder")
                sock.sendto(b"1", (UDP_IP, UDP_PORT))     # Enviar trigger

                # Guardar también en archivo de triggers
                writer_trig.writerow([now, line, "1"])

    except KeyboardInterrupt:
        print("\n--- Finalizó la lectura ---")
        print(f"Datos guardados en:   {FILE_LOG}")
        print(f"Triggers guardados en: {FILE_TRIGGERS}")
